# Level 3 — Business Analytics

## Objective

This notebook answers business questions using SQL queries and summarizes insights for stakeholders.

In [1]:
import pandas as pd
import duckdb

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

con = duckdb.connect()

con.execute("""
CREATE OR REPLACE VIEW superstore_features AS
SELECT *
FROM read_csv_auto('../data/superstore_features.csv');
""")

con.execute("""
SELECT *
FROM superstore_features
LIMIT 5
""").df()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,fulfillment_days,profit_margin,order_year,order_month,customer_lifetime_sales,customer_tier
0,2230,CA-2014-128055,2014-03-31,2014-04-05,Standard Class,AA-10315,Alex Avila,Consumer,United States,San Francisco,California,94122,West,OFF-BI-10004390,Office Supplies,Binders,GBC DocuBind 200 Manual Binding Machine,673.568,2,0.2,252.5880,5,0.3750,2014,3,5563.56,Premium
1,2231,CA-2014-128055,2014-03-31,2014-04-05,Standard Class,AA-10315,Alex Avila,Consumer,United States,San Francisco,California,94122,West,OFF-AP-10002765,Office Supplies,Appliances,Fellowes Advanced Computer Series Surge Protec...,52.980,2,0.0,14.8344,5,0.2800,2014,3,5563.56,Premium
2,5199,CA-2016-103982,2016-03-03,2016-03-08,Standard Class,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,78664,Central,OFF-SU-10000151,Office Supplies,Supplies,High Speed Automatic Electric Letter Opener,3930.072,3,0.2,-786.0144,5,-0.2000,2016,3,5563.56,Premium
3,5200,CA-2016-103982,2016-03-03,2016-03-08,Standard Class,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,78664,Central,OFF-FA-10001332,Office Supplies,Fasteners,"Acco Banker's Clasps, 5 3/4""-Long",2.304,1,0.2,0.7776,5,0.3375,2016,3,5563.56,Premium
4,5201,CA-2016-103982,2016-03-03,2016-03-08,Standard Class,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,78664,Central,TEC-PH-10000895,Technology,Phones,Polycom VVX 310 VoIP phone,431.976,3,0.2,32.3982,5,0.0750,2016,3,5563.56,Premium


## Selecting Variables for Business Analysis

The engineered dataset contains both the original retail variables and the features created in **Notebook 02**. While all variables remain available in the exported dataset, only those relevant to the business analyses in this notebook were selected.

The following columns were excluded for the reasons below:

* **`row_id`** and **`order_id`** were removed because they are unique identifiers and do not contribute to summary statistics or business insights.
* **`order_date`** was removed because the engineered features **`order_year`** and **`order_month`** provide the time granularity needed for this analysis.
* **`ship_date`** was removed because the engineered feature **`fulfillment_days`** more directly measures shipping performance.
* **`customer_id`** and **`product_id`** were removed in favor of **`customer_name`** and **`product_name`**, which produce more interpretable business reports.
* **`country`** was removed because every observation occurred in the United States, providing no additional analytical value.
* **`city`** and **`postal_code`** were removed because they contain **531** and **631** unique values, respectively. For this analysis, **`region`** and **`state`** provide a more meaningful level of geographic aggregation while reducing unnecessary granularity.

The resulting dataset retains the variables most relevant to analyzing customer behavior, product performance, geographic trends, profitability, and operational efficiency while keeping the analysis focused and easy to interpret.



In [2]:
analysis_df = con.execute("""
SELECT
    -- Customer
    customer_name,
    segment,

    -- Geography
    region,
    state,

    -- Product
    category,
    sub_category,
    product_name,

    -- Shipping
    ship_mode,

    -- Original Sales Metrics
    sales,
    quantity,
    discount,
    profit,

    -- Engineered Features
    order_year,
    order_month,
    fulfillment_days,
    profit_margin,
    customer_lifetime_sales,
    customer_tier

FROM superstore_features
""").df()

analysis_df.head().style.hide(axis="index")

customer_name,segment,region,state,category,sub_category,product_name,ship_mode,sales,quantity,discount,profit,order_year,order_month,fulfillment_days,profit_margin,customer_lifetime_sales,customer_tier
Alex Avila,Consumer,West,California,Office Supplies,Binders,GBC DocuBind 200 Manual Binding Machine,Standard Class,673.568000,2,0.200000,252.588000,2014,3,5,0.375000,5563.560000,Premium
Alex Avila,Consumer,West,California,Office Supplies,Appliances,Fellowes Advanced Computer Series Surge Protectors,Standard Class,52.980000,2,0.000000,14.834400,2014,3,5,0.280000,5563.560000,Premium
Alex Avila,Consumer,Central,Texas,Office Supplies,Supplies,High Speed Automatic Electric Letter Opener,Standard Class,3930.072000,3,0.200000,-786.014400,2016,3,5,-0.200000,5563.560000,Premium
Alex Avila,Consumer,Central,Texas,Office Supplies,Fasteners,"Acco Banker's Clasps, 5 3/4""-Long",Standard Class,2.304000,1,0.200000,0.777600,2016,3,5,0.337500,5563.560000,Premium
Alex Avila,Consumer,Central,Texas,Technology,Phones,Polycom VVX 310 VoIP phone,Standard Class,431.976000,3,0.200000,32.398200,2016,3,5,0.075000,5563.560000,Premium


## Customer Analysis

Customer analytics focuses on understanding purchasing behavior across individual customers and customer segments. By identifying high-value customers, comparing segment performance, and evaluating customer lifetime sales, we can better understand who generates the greatest value for the business and where customer retention efforts should be focused.

### Top Customers by Lifetime Sales

**Question:**

Who are our most valuable customers based on lifetime sales?

**Objective:**

Identify the customers who generate the greatest total revenue over their lifetime. Understanding which customers contribute the most sales helps businesses recognize high-value accounts and prioritize customer retention efforts.

**Variables:**

- `customer_name`
- `customer_lifetime_sales`

In [ ]:
con.execute("""
SELECT DISTINCT
    customer_name,
    customer_lifetime_sales
FROM superstore_features
ORDER BY customer_lifetime_sales DESC
LIMIT 10;
""").df().style.hide(axis="index").format({
    "customer_lifetime_sales": "${:,.2f}"
})

customer_name,customer_lifetime_sales
Sean Miller,"$25,043.05"
Tamara Chand,"$19,052.22"
Raymond Buch,"$15,117.34"
Tom Ashbrook,"$14,595.62"
Adrian Barton,"$14,473.57"
Ken Lonsdale,"$14,175.23"
Sanjit Chand,"$14,142.33"
Hunter Lopez,"$12,873.30"
Sanjit Engle,"$12,209.44"
Christopher Conant,"$12,129.07"


**Results:**

Sean Miller was the highest-value customer, generating **$25,043.05** in lifetime sales. Tamara Chand ranked second with **$19,052.22**, followed by Raymond Buch with **$15,117.34**. Together, the ten highest-value customers generated **$153,811.17** in lifetime sales.

**Key Insights:**

The results identify a small group of customers with substantial purchasing histories. Sean Miller generated approximately **$5,991 more** than the second-ranked customer, making him a particularly important account. These customers may be strong candidates for personalized promotions, loyalty rewards, and targeted retention efforts.

### Sales Performance by Customer Segment

**Question:**
Which customer segments generate the most revenue?

**Objective:**

Compare sales performance across the Consumer, Corporate, and Home Office segments to determine which customer groups contribute the greatest share of revenue. This analysis helps identify where the business generates the majority of its sales and whether certain segments warrant additional marketing or sales efforts.

**Variables:**
- `segment`
- `sales`

In [ ]:
con.execute("""
SELECT
    segment,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(
        100.0 * SUM(sales) / SUM(SUM(sales)) OVER (),
        2
    ) AS percentage_of_total_sales
FROM superstore_features
GROUP BY segment
ORDER BY total_sales DESC;
""").df().style.hide(axis="index").format({
    "total_sales": "${:,.2f}",
    "percentage_of_total_sales": "{:.2f}%"
})

segment,total_sales,percentage_of_total_sales
Consumer,"$1,161,401.34",50.56%
Corporate,"$706,146.37",30.74%
Home Office,"$429,653.15",18.70%


**Results:**

The Consumer segment generated the highest sales at **$1,161,401.34**, representing **50.56%** of total revenue. Corporate customers contributed **$706,146.37**, or **30.74%**, while the Home Office segment generated **$429,653.15**, accounting for the remaining **18.70%**.

**Key Insights:**

Consumer customers are the company’s largest source of revenue, producing slightly more than half of all sales. However, the Corporate segment also makes a substantial contribution, with Consumer and Corporate customers together accounting for **81.30%** of total sales. Although Home Office is the smallest segment, additional profitability analysis is needed before concluding that it is less valuable, since revenue alone does not account for costs or profit margins.

### Sales and Profitability by Customer Tier

**Question:**
How do sales and profitability differ across customer tiers?

**Objective:**
Compare total sales, total profit, and profit margins across customer tiers to determine whether higher-value customers also generate stronger profitability. This analysis helps distinguish customers who produce substantial revenue from those who create the greatest financial value for the business.

**Variables:**
`customer_tier`
`sales`
`profit`
`profit_margin`


In [ ]:
con.execute("""
SELECT
    customer_tier,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,
    100.0 * SUM(profit) / SUM(sales) AS profit_margin_percentage
FROM superstore_features
GROUP BY customer_tier
ORDER BY total_sales DESC;
""").df().style.hide(axis="index").format({
    "total_sales": "${:,.2f}",
    "total_profit": "${:,.2f}",
    "profit_margin_percentage": "{:.2f}%"
})


customer_tier,total_sales,total_profit,profit_margin_percentage
Big Fish,"$709,342.71","$122,167.79",17.22%
High Value,"$574,885.76","$52,925.88",9.21%
Premium,"$557,107.99","$65,687.41",11.79%
Standard,"$455,864.40","$45,615.94",10.01%


**Results:**

Big Fish customers generated the highest total sales at **$709,342.71** and the highest total profit at **$122,167.79**. They also achieved the strongest profit margin at **17.22%**.

High Value customers generated slightly more sales than Premium customers—**$574,885.76** compared with **$557,107.99**—but Premium customers produced considerably more profit: **$65,687.41** compared with **$52,925.88**. Premium customers also had a higher profit margin of **11.79%**, while the High Value tier had the lowest margin at **9.21%**.

**Key Insights:**

Customer value based on lifetime sales does not correspond perfectly with profitability. Although High Value customers generated more revenue than Premium customers, they produced approximately **$12,761 less profit** and had a profit margin that was **2.58 percentage points lower**.

Big Fish customers are especially valuable because they lead the other tiers in sales, total profit, and profit margin. The relatively weak margin among High Value customers warrants further investigation into factors such as discount usage, product mix, and shipping costs. Retention strategies should therefore consider profitability in addition to lifetime sales when identifying the company’s most valuable customers.

## Geographic Analysis

Geographic analysis evaluates business performance across regions and states. Comparing sales, profitability, and customer activity by location helps identify high-performing markets, uncover regional trends, and highlight areas that may benefit from targeted business strategies.

## Product Analysis

Product analysis examines sales and profitability across product categories, subcategories, and individual products. The objective is to identify top-performing products, recognize underperforming inventory, and understand which areas of the product portfolio contribute most to overall business success.

## Sales & Profitability Analysis

Sales and profitability analysis investigates the financial performance of the business by examining revenue, discounts, profit, and profit margins. This section explores how pricing and discounting strategies influence profitability and identifies opportunities to improve financial performance.

## Shipping & Time Analysis

Shipping and time analysis evaluates operational efficiency and temporal business trends. By analyzing fulfillment times, shipping methods, and sales performance across months and years, this section identifies seasonal patterns, monitors delivery performance, and uncovers trends that can support operational planning.

### Do Product Prices Differ Across Customer Segments?

Average unit prices and discounts were compared across customer segments to determine whether pricing patterns differed between Consumer, Corporate, and Home Office customers.

In [3]:
con.execute('''
SELECT 
    product_id,
    segment,
    ROUND(AVG(sales / quantity), 2) AS avg_unit_price,
    ROUND(AVG(discount), 3) AS avg_discount
FROM superstore_features
GROUP BY product_id, segment
ORDER BY product_id
Limit 40''').df()           

,product_id,segment,avg_unit_price,avg_discount
0,FUR-BO-10000112,Corporate,91.69,0.300
1,FUR-BO-10000330,Home Office,102.83,0.150
2,FUR-BO-10000330,Consumer,111.91,0.075
3,FUR-BO-10000362,Corporate,158.16,0.075
4,FUR-BO-10000362,Consumer,136.78,0.200
5,FUR-BO-10000362,Home Office,145.33,0.150
6,FUR-BO-10000468,Corporate,48.58,0.000
7,FUR-BO-10000468,Consumer,37.89,0.220
8,FUR-BO-10000711,Home Office,70.98,0.000
9,FUR-BO-10000711,Consumer,70.98,0.000


In [4]:
con.execute ("""
SELECT 
    segment,
    COUNT(DISTINCT customer_name) AS customer_count,
    ROUND(SUM(sales)) AS total_sales
FROM superstore_features
GROUP BY segment
""").df()


,segment,customer_count,total_sales
0,Consumer,409,1161401.0
1,Corporate,236,706146.0
2,Home Office,148,429653.0


In [5]:
con.execute('''
SELECT
    product_id,
    segment,
    ROUND(AVG(sales / quantity), 2) AS avg_unit_price,
    ROUND(AVG(discount), 3) AS avg_discount
FROM superstore_features
GROUP BY product_id, segment
ORDER BY product_id
LIMIT 40;
''').df()

,product_id,segment,avg_unit_price,avg_discount
0,FUR-BO-10000112,Corporate,91.69,0.300
1,FUR-BO-10000330,Consumer,111.91,0.075
2,FUR-BO-10000330,Home Office,102.83,0.150
3,FUR-BO-10000362,Corporate,158.16,0.075
4,FUR-BO-10000362,Consumer,136.78,0.200
5,FUR-BO-10000362,Home Office,145.33,0.150
6,FUR-BO-10000468,Corporate,48.58,0.000
7,FUR-BO-10000468,Consumer,37.89,0.220
8,FUR-BO-10000711,Home Office,70.98,0.000
9,FUR-BO-10000711,Consumer,70.98,0.000


In [6]:
con.execute('''
SELECT
    segment,
    ROUND(AVG(fulfillment_days),2) AS avg_fulfillment_days,
    MIN(fulfillment_days) AS min_days,
    MEDIAN(fulfillment_days) AS median_days,
    MAX(fulfillment_days) AS max_days
FROM superstore_features
GROUP BY segment
ORDER BY avg_fulfillment_days;
''').df()

,segment,avg_fulfillment_days,min_days,median_days,max_days
0,Home Office,3.92,0,4.0,7
1,Consumer,3.94,0,4.0,7
2,Corporate,4.01,0,4.0,7


In [7]:
con.execute("""
SELECT
    ship_mode,
    ROUND(AVG(fulfillment_days), 2) AS avg_fulfillment_days,
    MIN(fulfillment_days) AS min_days,
    MAX(fulfillment_days) AS max_days
FROM superstore_features
GROUP BY ship_mode
ORDER BY avg_fulfillment_days;
""").df()

,ship_mode,avg_fulfillment_days,min_days,max_days
0,Same Day,0.04,0,1
1,First Class,2.18,1,4
2,Second Class,3.24,1,5
3,Standard Class,5.01,3,7


extra stuff to think about: 

In [8]:
con.execute(''' 
SELECT
    product_name,
    sales,
    profit,
    profit_margin
FROM superstore_features
ORDER BY profit_margin
LIMIT 10
''').df() 

,product_name,sales,profit,profit_margin
0,Kensington 6 Outlet SmartSocket Surge Protector,24.588,-67.6170,-2.75
1,Eureka Disposable Bags for Sanitaire Vibra Gro...,1.624,-4.4660,-2.75
2,Hoover Portapower Portable Vacuum,2.688,-7.3920,-2.75
3,Hoover Shoulder Vac Commercial Portable Vacuum,143.128,-393.6020,-2.75
4,Euro Pro Shark Stick Mini Vacuum,48.784,-131.7168,-2.70
5,Acco Smartsocket Color-Coded Six-Outlet AC Ada...,26.406,-71.2962,-2.70
6,Tripp Lite Isotel 8 Ultra 8 Outlet Metal Surge,70.970,-191.6190,-2.70
7,Fellowes 8 Outlet Superior Workstation Surge P...,33.620,-90.7740,-2.70
8,Belkin 6 Outlet Metallic Surge Strip,4.356,-11.7612,-2.70
9,Hoover Commercial Lightweight Upright Vacuum,1.392,-3.7584,-2.70
